# Multiprocessor Sequential IDK Cascade (REAL TIME CLASSIFICATION)

This notebook runs a pipelined sequential IDK cascade with ResNet-18, ResNet-34, ResNet-50, and ResNet-152.
Each ResNet model runs in its own MPS process, so an earlier model can start the next input after it routes an IDK sample downstream.


This cell imports the packages and finds the repo paths used by the notebook.


In [10]:
from collections import Counter, deque
from pathlib import Path
import sys
import tarfile
import time

import numpy as np
import torch
import torch.multiprocessing as mp
from PIL import Image
from torchvision import transforms

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SCRIPTS_DIR = PROJECT_ROOT / "scripts"
IMAGENETV2_DIR = PROJECT_ROOT / "ImageNet-V2 DataSet"

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from real_time_mp_workers import model_worker


This cell sets the models, test dataset, and confidence threshold.


In [11]:
MODEL_A = "resnet18"
MODEL_B = "resnet34"
MODEL_C = "resnet50"
MODEL_D = "resnet152"
MODELS = (MODEL_A, MODEL_B, MODEL_C, MODEL_D)

VARIANT_ARCHIVES = {
    "matched-frequency": IMAGENETV2_DIR / "imagenetv2-matched-frequency.tar.gz",
    "threshold-0.7": IMAGENETV2_DIR / "imagenetv2-threshold0.7.tar.gz",
    "top-images": IMAGENETV2_DIR / "imagenetv2-top-images.tar.gz",
}

TEST_VARIANT = "threshold-0.7"
TEST_SAMPLES = 10000
CLASSIFICATION_THRESHOLD = 0.7
MAX_IN_FLIGHT_BY_MODEL = {model_name: 1 for model_name in MODELS}

if not torch.backends.mps.is_available():
    raise RuntimeError("MPS is required for this multiprocessor notebook")

DEVICE_BY_MODEL = {model_name: "mps" for model_name in MODELS}


This cell streams images and labels directly from the local ImageNet-V2 tar files.


In [12]:
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")


def label_from_key(key):
    return int(key.split("/")[1])


def stream_rows(variant, max_samples):
    emitted = 0
    with tarfile.open(VARIANT_ARCHIVES[variant], "r:*") as tar:
        for member in tar:
            if not member.isfile() or not member.name.lower().endswith(IMAGE_EXTENSIONS):
                continue

            image_file = tar.extractfile(member)
            image = Image.open(image_file).convert("RGB")
            image_file.close()

            yield image, label_from_key(member.name)
            emitted += 1
            if emitted >= max_samples:
                break


This cell prepares ImageNet preprocessing and starts one MPS worker process per ResNet model.


In [13]:
preprocess = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ]
)


def image_to_batch(image):
    return preprocess(image).unsqueeze(0)


def start_model_processes():
    mp.set_start_method("spawn", force=True)
    ctx = mp.get_context("spawn")
    job_queues = {model_name: ctx.SimpleQueue() for model_name in MODELS}
    result_queue = ctx.SimpleQueue()
    processes = []

    for model_name in MODELS:
        process = ctx.Process(
            target=model_worker,
            args=(model_name, DEVICE_BY_MODEL[model_name], job_queues[model_name], result_queue),
        )
        process.start()
        processes.append(process)

    return job_queues, result_queue, processes


def stop_model_processes(job_queues, processes):
    for queue in job_queues.values():
        queue.put(None)
    for process in processes:
        process.join()


This cell defines prediction and IDK helpers for worker-process inference.


In [14]:
def is_idk(confidence):
    return confidence < CLASSIFICATION_THRESHOLD


def predict_in_worker(model_name, sample_index, image_tensor, job_queues, result_queue):
    job_queues[model_name].put((sample_index, image_tensor))
    message = result_queue.get()

    if message[0] == "error":
        _, worker_name, error = message
        raise RuntimeError(f"{worker_name} worker failed: {error}")

    result_sample_index, result_model_name, probabilities, prediction, confidence, elapsed_ms = message
    if result_sample_index != sample_index or result_model_name != model_name:
        raise RuntimeError(f"Unexpected worker result from {result_model_name} for sample {result_sample_index}")

    return probabilities, prediction, confidence, elapsed_ms


This cell defines pipeline routing helpers for the sequential IDK chain.


In [15]:
NEXT_MODEL_BY_MODEL = {MODELS[i]: MODELS[i + 1] for i in range(len(MODELS) - 1)}


def route_to_next_model(model_name, confidence):
    if model_name == MODELS[-1] or not is_idk(confidence):
        return None
    return NEXT_MODEL_BY_MODEL[model_name]


def model_backlog_size(model_name, pending_by_model, in_flight_by_model):
    return len(pending_by_model[model_name]) + in_flight_by_model[model_name]


This cell names the cascade and prints the process placement.


In [16]:
cascade_name = "multiprocessor_sequential_idk"

print("Cascade:", cascade_name)
print("Cascade order:", MODELS)
print("Device by model:", DEVICE_BY_MODEL)
print("Max in flight by model:", MAX_IN_FLIGHT_BY_MODEL)


Cascade: multiprocessor_sequential_idk
Cascade order: ('resnet18', 'resnet34', 'resnet50', 'resnet152')
Device by model: {'resnet18': 'mps', 'resnet34': 'mps', 'resnet50': 'mps', 'resnet152': 'mps'}
Max in flight by model: {'resnet18': 1, 'resnet34': 1, 'resnet50': 1, 'resnet152': 1}


This cell runs the pipelined sequential cascade on the test split and stores the metrics.


In [17]:
def run_multiprocessor_sequential_idk_cascade():
    job_queues, result_queue, processes = start_model_processes()

    try:
        labels = np.full(TEST_SAMPLES, -1, dtype=np.int64)
        final_predictions = np.full(TEST_SAMPLES, -1, dtype=np.int64)
        chosen_models = np.full(TEST_SAMPLES, "", dtype="<U32")
        latencies_ms = np.full(TEST_SAMPLES, np.nan, dtype=np.float64)

        sample_tensors = {}
        sample_start_times = {}
        pending_by_model = {model_name: deque() for model_name in MODELS}
        in_flight_by_model = Counter()
        execution_count_by_model = Counter()
        execution_time_ms_by_model = Counter()
        final_count_by_model = Counter()
        idk_count_by_model = Counter()
        route_count_to_model = Counter()
        queue_max_size_by_model = Counter()

        row_iterator = iter(stream_rows(TEST_VARIANT, TEST_SAMPLES))
        next_sample_index = 0
        completed_sample_count = 0
        input_exhausted = False
        run_start = time.perf_counter()

        def enqueue_model(model_name, sample_index):
            pending_by_model[model_name].append(sample_index)
            queue_max_size_by_model[model_name] = max(
                queue_max_size_by_model[model_name],
                len(pending_by_model[model_name]),
            )

        def dispatch_model(model_name):
            while (
                pending_by_model[model_name]
                and in_flight_by_model[model_name] < MAX_IN_FLIGHT_BY_MODEL[model_name]
            ):
                sample_index = pending_by_model[model_name].popleft()
                job_queues[model_name].put((sample_index, sample_tensors[sample_index]))
                in_flight_by_model[model_name] += 1
                execution_count_by_model[model_name] += 1

        def dispatch_all_models():
            for model_name in MODELS:
                dispatch_model(model_name)

        def load_next_inputs():
            nonlocal input_exhausted, next_sample_index
            while (
                not input_exhausted
                and next_sample_index < TEST_SAMPLES
                and model_backlog_size(MODEL_A, pending_by_model, in_flight_by_model) < MAX_IN_FLIGHT_BY_MODEL[MODEL_A]
            ):
                try:
                    image, label = next(row_iterator)
                except StopIteration:
                    input_exhausted = True
                    break

                sample_index = next_sample_index
                next_sample_index += 1
                labels[sample_index] = int(label)
                sample_tensors[sample_index] = image_to_batch(image)
                sample_start_times[sample_index] = time.perf_counter()
                enqueue_model(MODEL_A, sample_index)

            if next_sample_index >= TEST_SAMPLES:
                input_exhausted = True

        def finish_sample(sample_index, model_name, prediction):
            nonlocal completed_sample_count
            final_predictions[sample_index] = prediction
            chosen_models[sample_index] = model_name
            final_count_by_model[model_name] += 1
            latencies_ms[sample_index] = (time.perf_counter() - sample_start_times.pop(sample_index)) * 1000.0
            sample_tensors.pop(sample_index, None)
            completed_sample_count += 1

        while not input_exhausted or completed_sample_count < next_sample_index:
            load_next_inputs()
            dispatch_all_models()

            if sum(in_flight_by_model.values()) == 0:
                if input_exhausted:
                    break
                continue

            message = result_queue.get()
            if message[0] == "error":
                _, worker_name, error = message
                raise RuntimeError(f"{worker_name} worker failed: {error}")

            sample_index, model_name, _, prediction, confidence, elapsed_ms = message
            in_flight_by_model[model_name] -= 1
            execution_time_ms_by_model[model_name] += elapsed_ms

            if is_idk(confidence):
                idk_count_by_model[model_name] += 1

            next_model = route_to_next_model(model_name, confidence)
            if next_model is None:
                finish_sample(sample_index, model_name, prediction)
            else:
                route_count_to_model[next_model] += 1
                enqueue_model(next_model, sample_index)

        labels = labels[:next_sample_index]
        final_predictions = final_predictions[:next_sample_index]
        chosen_models = chosen_models[:next_sample_index]
        latencies_ms = latencies_ms[:next_sample_index]
        total_wall_time_seconds = time.perf_counter() - run_start
        correct_predictions = int(np.count_nonzero(final_predictions == labels))

        return {
            "total_samples": int(len(labels)),
            "accuracy": float(correct_predictions / len(labels)),
            "correct_predictions": correct_predictions,
            "total_wall_time_seconds": float(total_wall_time_seconds),
            "throughput_fps": float(len(labels) / total_wall_time_seconds),
            "mean_latency_ms": float(latencies_ms.mean()),
            "cascade_name": cascade_name,
            "cascade_order": list(MODELS),
            "device_by_model": DEVICE_BY_MODEL,
            "max_in_flight_by_model": MAX_IN_FLIGHT_BY_MODEL,
            "final_prediction_count_by_model": {model: int(final_count_by_model[model]) for model in MODELS},
            "execution_count_by_model": {model: int(execution_count_by_model[model]) for model in MODELS},
            "mean_execution_time_ms_by_model": {
                model: float(execution_time_ms_by_model[model] / execution_count_by_model[model])
                if execution_count_by_model[model]
                else 0.0
                for model in MODELS
            },
            "idk_count_by_model": {model: int(idk_count_by_model[model]) for model in MODELS},
            "route_count_to_model": {model: int(route_count_to_model[model]) for model in MODELS},
            "queue_max_size_by_model": {model: int(queue_max_size_by_model[model]) for model in MODELS},
        }

    finally:
        stop_model_processes(job_queues, processes)


results = run_multiprocessor_sequential_idk_cascade()


This cell prints the relevant metrics for the multiprocessor sequential run.


In [18]:
print("Real-Time Multiprocessor Sequential MPS Test")
print("Confidence Threshold Constant: ", CLASSIFICATION_THRESHOLD)
print("Cascade:", results["cascade_name"])
print("Cascade order:", results["cascade_order"])
print("Device by model:", results["device_by_model"])
print("Max in flight by model:", results["max_in_flight_by_model"])
print("Total samples:", results["total_samples"])
print("Accuracy:", round(results["accuracy"], 4))
print("Correct predictions:", results["correct_predictions"])
print("Total wall time (seconds):", round(results["total_wall_time_seconds"], 3))
print("Throughput (FPS):", round(results["throughput_fps"], 3))
print("Mean latency (ms):", round(results["mean_latency_ms"], 3))
print()
print("Final prediction count by model:")
for model_name in MODELS:
    print(f"  {model_name}: {results['final_prediction_count_by_model'][model_name]}")
print("Execution count by model:")
for model_name in MODELS:
    print(f"  {model_name}: {results['execution_count_by_model'][model_name]}")
print("IDK count by model:")
for model_name in MODELS:
    print(f"  {model_name}: {results['idk_count_by_model'][model_name]}")
print("Route count to model:")
for model_name in MODELS:
    print(f"  {model_name}: {results['route_count_to_model'][model_name]}")
print("Max pending queue size by model:")
for model_name in MODELS:
    print(f"  {model_name}: {results['queue_max_size_by_model'][model_name]}")
print("Mean execution time by model (ms):")
for model_name in MODELS:
    print(f"  {model_name}: {results['mean_execution_time_ms_by_model'][model_name]:.3f}")


Real-Time Multiprocessor Sequential MPS Test
Confidence Threshold Constant:  0.7
Cascade: multiprocessor_sequential_idk
Cascade order: ['resnet18', 'resnet34', 'resnet50', 'resnet152']
Device by model: {'resnet18': 'mps', 'resnet34': 'mps', 'resnet50': 'mps', 'resnet152': 'mps'}
Max in flight by model: {'resnet18': 1, 'resnet34': 1, 'resnet50': 1, 'resnet152': 1}
Total samples: 10000
Accuracy: 0.7684
Correct predictions: 7684
Total wall time (seconds): 275.509
Throughput (FPS): 36.296
Mean latency (ms): 8606.355

Final prediction count by model:
  resnet18: 5555
  resnet34: 1374
  resnet50: 64
  resnet152: 3007
Execution count by model:
  resnet18: 10000
  resnet34: 4445
  resnet50: 3071
  resnet152: 3007
IDK count by model:
  resnet18: 4445
  resnet34: 3071
  resnet50: 3007
  resnet152: 2247
Route count to model:
  resnet18: 0
  resnet34: 4445
  resnet50: 3071
  resnet152: 3007
Max pending queue size by model:
  resnet18: 1
  resnet34: 8
  resnet50: 4
  resnet152: 612
Mean execution t